# Module 12: Data Cleaning & Exploratory Data Analysis

**Duration:** 20 hours  
**ML Focus:** Cleaning Titanic & Ames Housing for Modeling

Data cleaning and EDA represent 60-80% of a data scientist's workflow. This module provides a systematic framework: assess, clean, validate, explore.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import IsolationForest

%matplotlib inline
sns.set_theme()
print('Libraries loaded.')

## 1. Missing Data Patterns: MCAR, MAR, MNAR

Understanding *why* data is missing determines *how* you should handle it.

In [ ]:
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv(url)

print('=== Missing Value Analysis ===')
missing = pd.DataFrame({
    'Count': titanic.isna().sum(),
    'Percentage': (titanic.isna().sum() / len(titanic) * 100).round(2)
}).query('Count > 0')
print(missing)

print('\n=== Classifying Missing Mechanisms ===')
print('Cabin (77% missing): LIKELY MNAR')
print('  - Reason: Cabins were not recorded for most passengers, possibly because')
print('    those with cheaper tickets (lower classes) were not assigned cabins')
print('  - Missing is related to the value itself + correlated with Pclass')
print('\nAge (20% missing): LIKELY MAR')
print('  - Reason: Age may not have been recorded for certain groups')
print('  - Children (lower priority for record-keeping) may have more missing')
print('  - Missingness depends on observed data (Pclass, SibSp)')
print('\nEmbarked (0.2% missing): LIKELY MCAR')
print('  - Reason: Only 2 missing, likely random oversight in data entry')

# Visualize missing patterns
try:
    import missingno as msno
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    msno.matrix(titanic, ax=axes[0])
    msno.heatmap(titanic, ax=axes[1])
    plt.tight_layout()
    plt.show()
except ImportError:
    print('Install missingno: pip install missingno')

## 2. Handling Missing Data Strategically

Different strategies for different columns based on missing mechanism and importance.

In [ ]:
titanic_clean = titanic.copy()

# Strategy 1: Age — Group median imputation (MAR -> conditioned on observed)
age_medians = titanic_clean.groupby(['Pclass', 'Sex'])['Age'].median()
print('Age medians by Pclass and Sex:')
print(age_medians)

titanic_clean['Age'] = titanic_clean.groupby(['Pclass', 'Sex'])['Age'].transform(
    lambda x: x.fillna(x.median())
)
print(f'\nMissing Age after imputation: {titanic_clean["Age"].isna().sum()}')

# Strategy 2: Embarked — Mode imputation (MCAR -> simple fill is fine)
embarked_mode = titanic_clean['Embarked'].mode()[0]
titanic_clean['Embarked'].fillna(embarked_mode, inplace=True)
print(f'Missing Embarked after: {titanic_clean["Embarked"].isna().sum()}')

# Strategy 3: Cabin — Extract information then drop
titanic_clean['Deck'] = titanic_clean['Cabin'].str.extract(r'([A-Z])', expand=False)
titanic_clean.drop(columns=['Cabin'], inplace=True)
print(f'Deck missing values: {titanic_clean["Deck"].isna().sum()} (expected - kept as NaN for modeling)')

print('\n=== Missing values after cleaning ===')
print(titanic_clean.isna().sum().to_string())

## 3. Outlier Detection: IQR, Z-Score, and Isolation Forest

Multiple methods should be used together for robust outlier detection.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Method 1: IQR
Q1 = titanic_clean['Fare'].quantile(0.25)
Q3 = titanic_clean['Fare'].quantile(0.75)
IQR = Q3 - Q1
lower_iqr = Q1 - 1.5 * IQR
upper_iqr = Q3 + 1.5 * IQR
iqr_outliers = titanic_clean[(titanic_clean['Fare'] < lower_iqr) | (titanic_clean['Fare'] > upper_iqr)]

axes[0].boxplot(titanic_clean['Fare'])
axes[0].axhline(y=upper_iqr, color='r', linestyle='--', label=f'Upper fence: {upper_iqr:.0f}')
axes[0].set_title(f'IQR Method: {len(iqr_outliers)} outliers')
axes[0].set_ylabel('Fare')
axes[0].legend()

# Method 2: Z-score
z_scores = np.abs(stats.zscore(titanic_clean['Fare']))
z_outliers = titanic_clean[z_scores > 3]

axes[1].hist(z_scores, bins=50, edgecolor='white')
axes[1].axvline(x=3, color='r', linestyle='--', label='Z=3 threshold')
axes[1].set_title(f'Z-Score Method: {len(z_outliers)} outliers')
axes[1].set_xlabel('Z-Score')
axes[1].set_ylabel('Count')
axes[1].legend()

# Method 3: Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
iso_labels = iso.fit_predict(titanic_clean[['Fare', 'Age']])
iso_outliers = titanic_clean[iso_labels == -1]

colors = ['red' if l == -1 else 'blue' for l in iso_labels]
axes[2].scatter(titanic_clean['Age'], titanic_clean['Fare'], c=colors, alpha=0.5, s=10)
axes[2].set_title(f'Isolation Forest: {len(iso_outliers)} outliers')
axes[2].set_xlabel('Age')
axes[2].set_ylabel('Fare')

plt.tight_layout()
plt.show()

print(f'Overlap between IQR and Z-score: {len(set(iqr_outliers.index) & set(z_outliers.index))}')
print('\nHandling: Capping Fare at 99th percentile')
fare_upper = titanic_clean['Fare'].quantile(0.99)
titanic_clean['Fare'] = titanic_clean['Fare'].clip(upper=fare_upper)
print(f'Fare capped at: {fare_upper:.2f}')

## 4. Duplicates & Inconsistent Data

Real-world data is messy — whitespace, casing, typos, and duplicates are common.

In [ ]:
print('=== Duplicate Check ===')
print(f'Exact duplicate rows: {titanic_clean.duplicated().sum()}')
print(f'Duplicate PassengerIds: {titanic_clean["PassengerId"].duplicated().sum()}')

print('\n=== Inconsistent Data Check ===')
print('Sex values:', titanic_clean['Sex'].unique())
print('Embarked values:', titanic_clean['Embarked'].unique())

# Simulate some messy data
messy = titanic_clean.copy()
messy.loc[0, 'Sex'] = ' male '  # leading space
messy.loc[1, 'Sex'] = 'MALE'    # uppercase
messy.loc[2, 'Embarked'] = 's '  # trailing space

print('\nSimulated messy values after adding noise:')
print('Sex:', messy['Sex'].unique())
print('Embarked:', messy['Embarked'].unique())

# Cleaning
messy['Sex'] = messy['Sex'].str.strip().str.lower().str.capitalize()
messy['Embarked'] = messy['Embarked'].str.strip().str.upper()
print('\nAfter cleaning:')
print('Sex:', messy['Sex'].unique())
print('Embarked:', messy['Embarked'].unique())

## 5. Data Validation

Always validate your data after cleaning — check ranges, types, and constraints.

In [ ]:
print('=== Data Validation ===')
validations = []

# Type checks
validations.append(('Pclass is int', titanic_clean['Pclass'].dtype == 'int64'))
validations.append(('Survived is int', titanic_clean['Survived'].dtype == 'int64'))

# Range checks
validations.append(('Age 0-120', titanic_clean['Age'].between(0, 120).all()))
validations.append(('Fare >= 0', (titanic_clean['Fare'] >= 0).all()))
validations.append(('Pclass in 1-3', titanic_clean['Pclass'].isin([1, 2, 3]).all()))
validations.append(('Survived in 0-1', titanic_clean['Survived'].isin([0, 1]).all()))

# Uniqueness
validations.append(('Unique PassengerId', titanic_clean['PassengerId'].is_unique))

# No missing
validations.append(('No missing values', titanic_clean.isna().sum().sum() == 0))

for check, result in validations:
    status = 'PASS' if result else 'FAIL'
    print(f'  [{status}] {check}')

if all(r for _, r in validations):
    print('\nAll validations passed! Dataset is clean.')

## 6. EDA Framework: Univariate, Bivariate, Multivariate

A structured EDA framework ensures you don't miss important insights.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Univariate: Numeric
numeric_cols = ['Age', 'Fare', 'SibSp', 'Parch']
for i, (col, ax) in enumerate(zip(numeric_cols, axes.flatten()[:4])):
    sns.histplot(titanic_clean[col], bins=30, kde=True, ax=ax)
    ax.set_title(f'{col} (Skew: {titanic_clean[col].skew():.2f})')

# Univariate: Categorical
sns.countplot(data=titanic_clean, x='Pclass', ax=axes[0, 2])
axes[0, 2].set_title('Pclass Distribution')

sns.countplot(data=titanic_clean, x='Sex', ax=axes[1, 2])
axes[1, 2].set_title('Sex Distribution')

plt.tight_layout()
plt.show()

print('=== Bivariate: Correlation with Target (Survived) ===')
numeric_df = titanic_clean.select_dtypes(include=[np.number])
corr_with_target = numeric_df.corr()['Survived'].drop('Survived').sort_values(ascending=False)
print(corr_with_target.round(3))

print('\n=== Key Insights ===')
print('1. Pclass and Sex are the strongest predictors of survival')
print('2. Fare (proxy for class) is moderately correlated')
print('3. Family size has a non-linear relationship')

## 7. Correlation Analysis: Pearson, Spearman, and Categorical

Different correlation measures for different data types.

In [ ]:
num_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']

print('=== Pearson Correlation (Linear) ===')
pearson = titanic_clean[num_cols].corr(method='pearson')
print(pearson.round(3))

print('\n=== Spearman Correlation (Monotonic) ===')
spearman = titanic_clean[num_cols].corr(method='spearman')
print(spearman.round(3))

print('\n=== Difference (Spearman - Pearson) ===')
print((spearman - pearson).round(3))

print('\nLarge differences indicate non-linear relationships.')

# Categorical correlation: Cramers V for categorical pairs
print('\n=== Cross-tab: Sex vs Survived ===')
ct = pd.crosstab(titanic_clean['Sex'], titanic_clean['Survived'])
print(ct)
print('\nSurvival rate by Sex:')
print(titanic_clean.groupby('Sex')['Survived'].mean().round(3))

## 8. Automated EDA with ydata-profiling

Automated EDA tools provide a comprehensive overview in seconds.

In [ ]:
print('=== Automated EDA with ydata-profiling ===')
print('Run the following to generate an HTML report:')
print()
print('from ydata_profiling import ProfileReport')
print("profile = ProfileReport(titanic_clean, title='Titanic EDA Report', explorative=True)")
print("profile.to_file('titanic_eda_report.html')")
print()
print('The report includes:')
print('  - Overview: dataset statistics, warnings')
print('  - Variables: detailed analysis per column')
print('  - Correlations: heatmaps for multiple methods')
print('  - Missing values: matrix and bar chart')
print('  - Sample: first 10 rows of the data')
print()
try:
    from ydata_profiling import ProfileReport
    profile = ProfileReport(titanic_clean, title='Titanic EDA', explorative=True, minimal=True)
    print('Profile report generated in memory.')
except ImportError:
    print('Install with: pip install ydata-profiling')

## 9. ML Focus: Cleaning Impact on Model Performance

Let's quantify how much cleaning improves model performance.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

print('=== Impact of Cleaning on Model Performance ===')

# Raw data baseline (drop all NaN, basic encoding)
raw = titanic.copy()
raw = raw.dropna().drop(columns=['Name', 'Ticket', 'Cabin', 'PassengerId'])
raw['Sex'] = (raw['Sex'] == 'female').astype(int)
raw = pd.get_dummies(raw, columns=['Embarked'], drop_first=True, dtype=int)

X_raw = raw.drop(columns=['Survived'])
y_raw = raw['Survived']

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])
scores_raw = cross_val_score(pipe, X_raw, y_raw, cv=5, scoring='accuracy')
print(f'Raw data (dropped NaN): {scores_raw.mean():.4f} +/- {scores_raw.std():.4f}')

# Cleaned data (imputed, engineered, validated)
clean = titanic_clean.copy()
clean = clean.drop(columns=['Name', 'Ticket', 'PassengerId', 'Deck'])
clean['Sex'] = (clean['Sex'] == 'female').astype(int)
clean = pd.get_dummies(clean, columns=['Embarked'], drop_first=True, dtype=int)

X_clean = clean.drop(columns=['Survived'])
y_clean = clean['Survived']

scores_clean = cross_val_score(pipe, X_clean, y_clean, cv=5, scoring='accuracy')
print(f'Cleaned data (imputed): {scores_clean.mean():.4f} +/- {scores_clean.std():.4f}')

print(f'\nImprovement: {(scores_clean.mean() - scores_raw.mean()) * 100:.2f} percentage points')
print(f'More rows used: {len(X_clean) - len(X_raw)} additional passengers')

## 10. Complete Data Cleaning Pipeline (Reusable)

A well-structured cleaning function that can be reused.

In [ ]:
def clean_titanic_pipeline(df):
    """Complete, reusable Titanic data cleaning pipeline."""
    data = df.copy()
    
    # Step 1: Missing Data
    data['Age'] = data.groupby(['Pclass', 'Sex'])['Age'].transform(
        lambda x: x.fillna(x.median())
    )
    data['Embarked'].fillna(data['Embarked'].mode()[0], inplace=True)
    data['Deck'] = data['Cabin'].str[0]
    data.drop(columns=['Cabin'], inplace=True)
    
    # Step 2: Feature Engineering
    data['Family_Size'] = data['SibSp'] + data['Parch'] + 1
    data['Is_Alone'] = (data['Family_Size'] == 1).astype(int)
    data['Title'] = data['Name'].str.extract(r',\s*([^\.]+)\.', expand=False)
    rare_titles = ['Don', 'Rev', 'Dr', 'Mme', 'Ms', 'Major', 'Lady', 'Sir', 'Mlle', 'Col', 'Capt', 'Countess', 'Jonkheer']
    data['Title'] = data['Title'].apply(lambda x: 'Rare' if x in rare_titles else x)
    
    # Step 3: Outlier Capping
    fare_upper = data['Fare'].quantile(0.99)
    data['Fare'] = data['Fare'].clip(upper=fare_upper)
    
    # Step 4: Consistent Formatting
    data['Sex'] = data['Sex'].str.strip().str.lower().str.capitalize()
    data['Embarked'] = data['Embarked'].str.strip().str.upper()
    
    # Step 5: Drop unnecessary columns
    data.drop(columns=['PassengerId', 'Name', 'Ticket'], inplace=True, errors='ignore')
    
    # Step 6: Validate
    assert data['Age'].between(0, 120).all(), 'Age validation failed'
    assert (data['Fare'] >= 0).all(), 'Fare validation failed'
    assert data['Pclass'].isin([1, 2, 3]).all(), 'Pclass validation failed'
    
    return data

# Test the pipeline
original = pd.read_csv(url)
cleaned = clean_titanic_pipeline(original)
print('Pipeline test complete!')
print('Shape:', cleaned.shape)
print('Missing values:', cleaned.isna().sum().sum())
print('\nModule 12 complete! You now have a systematic approach to data cleaning and EDA.')